In [ ]:
import numpy as np, pandas as pd, scipy.stats as stats
import pyarrow as pa, pyarrow.parquet as pq
import re, shutil, time, glob, io, os, zipfile, requests
import statsmodels.formula.api as smf
from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from statsmodels.stats.outliers_influence import variance_inflation_factor

BASE_DIR   = Path("")
 
RAW_DIR = BASE_DIR / "Data" / "Raw"
INTERIM_DIR = BASE_DIR / "Data" / "Interim"
PROCESSED_DIR = BASE_DIR / "Data" / "Processed"

MERGED_DIR = BASE_DIR / "Data" / "Merged" 
MERGED_DIR.mkdir(parents=True, exist_ok=True) 

FINAL_DIR = BASE_DIR / "Data" / "Final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Merge

## create prac*YEAR_MONTH_DATE*sub grid

In [4]:
eprac = pd.read_parquet(PROCESSED_DIR / "epraccur_processed.parquet")
bnf = pd.read_csv(PROCESSED_DIR / "bnf_processed.csv")

In [5]:
# Practices already filtered in the epraccur cell (England, active in window, GP practices)
in_scope = pd.read_parquet(PROCESSED_DIR / "epraccur_processed.parquet")
assert not in_scope["PRACTICE_CODE"].duplicated().any()

months = pd.date_range("2022-07-01", "2026-03-01", freq="MS")   # 45 months, no April 2026

CODE, NAME = "BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"

# --- one code and one name for apomorphine hydrochloride ---
bnf = bnf.copy()
bnf[CODE] = bnf[CODE].astype(str).str.strip().replace({"0409010AC": "0409010A0"})
bnf.loc[bnf[CODE] == "0409010A0", NAME] = "Apomorphine hydrochloride"

subs = (
    bnf[[CODE, NAME, "BNF_PARAGRAPH_CODE", "BNF_PARAGRAPH"]]
    .drop_duplicates()
    .sort_values(CODE)
)
subs["BNF_PARAGRAPH_CODE"] = subs["BNF_PARAGRAPH_CODE"].astype(str).str.zfill(6)

prac_month = in_scope.merge(pd.DataFrame({"YEAR_MONTH_DATE": months}), how="cross")
month_end = prac_month["YEAR_MONTH_DATE"] + pd.offsets.MonthEnd(0)
prac_month = prac_month[
    (prac_month["OPEN_DATE"] <= month_end)
    & (prac_month["CLOSE_DATE"].isna() | (prac_month["CLOSE_DATE"] >= prac_month["YEAR_MONTH_DATE"]))
]

grid = (
    prac_month.merge(subs, how="cross")
    .sort_values(["PRACTICE_CODE", "YEAR_MONTH_DATE", CODE])
    .reset_index(drop=True)
)

EXPECTED_SUBS = 26     # set to the number you expect: see the note below
assert subs[CODE].is_unique, subs[subs[CODE].duplicated(keep=False)]    # shows any code with 2 names/paragraphs
assert "0409010AC" not in set(subs[CODE])
assert len(subs) == EXPECTED_SUBS, f"{len(subs)} substances"
assert len(months) == 45
assert months.max() == pd.Timestamp("2026-03-01")
assert len(grid) == len(prac_month) * len(subs)
assert not grid.duplicated(["PRACTICE_CODE", "YEAR_MONTH_DATE", CODE]).any()
print(f"{in_scope['PRACTICE_CODE'].nunique():,} practices | {len(months)} months | {len(subs)} substances | grid rows: {len(grid):,}")

6,639 practices | 45 months | 26 substances | grid rows: 7,527,546


In [6]:
grid.head()

,PRACTICE_CODE,PRACTICE_NAME,ICB_CODE,POSTCODE,OPEN_DATE,CLOSE_DATE,STATUS,COMMISSIONER,PRESCRIBING_SETTING,YEAR_MONTH_DATE,BNF_CHEMICAL_SUBSTANCE_CODE,BNF_CHEMICAL_SUBSTANCE,BNF_PARAGRAPH_CODE,BNF_PARAGRAPH
0,A81001,THE DENSHAM SURGERY,QHM,TS18 1HU,1974-04-01,NaT,ACTIVE,16C,RO76,2022-07-01,0409010A0,Apomorphine hydrochloride,040901,Dopaminergic drugs used in parkinsonism
1,A81001,THE DENSHAM SURGERY,QHM,TS18 1HU,1974-04-01,NaT,ACTIVE,16C,RO76,2022-07-01,0409010AA,Safinamide,040901,Dopaminergic drugs used in parkinsonism
2,A81001,THE DENSHAM SURGERY,QHM,TS18 1HU,1974-04-01,NaT,ACTIVE,16C,RO76,2022-07-01,0409010AB,Opicapone,040901,Dopaminergic drugs used in parkinsonism
3,A81001,THE DENSHAM SURGERY,QHM,TS18 1HU,1974-04-01,NaT,ACTIVE,16C,RO76,2022-07-01,0409010B0,Amantadine hydrochloride,040901,Dopaminergic drugs used in parkinsonism
4,A81001,THE DENSHAM SURGERY,QHM,TS18 1HU,1974-04-01,NaT,ACTIVE,16C,RO76,2022-07-01,0409010D0,Carbidopa,040901,Dopaminergic drugs used in parkinsonism


## ons + imd

In [7]:
# ---- Load the processed lookups ----
ons = pd.read_parquet(
    PROCESSED_DIR / "onspd_processed.parquet",
    columns=["pcds", "ruc21ind", "sicbl26cd", "msoa21cd", "lsoa21cd"],
).rename(columns={"pcds": "POSTCODE"})
assert ons["POSTCODE"].is_unique

imd_small = pd.read_parquet(
    PROCESSED_DIR / "imd2025_processed.parquet",
    columns=["lsoa21cd", "imd25_rank", "imd25_decile"],
)
assert imd_small["lsoa21cd"].is_unique

# ---- Practice-level lookup ----
prac = in_scope[["PRACTICE_CODE", "POSTCODE"]].drop_duplicates()
prac = (
    prac.merge(ons, on="POSTCODE", how="left", validate="m:1")
        .merge(imd_small, on="lsoa21cd", how="left", validate="m:1")
        .drop(columns="POSTCODE")
)
assert prac["PRACTICE_CODE"].is_unique

# ---- Merge onto grid (drop old copies first to avoid _x/_y suffixes) ----
new_cols = [c for c in prac.columns if c != "PRACTICE_CODE"]
grid = grid.drop(columns=[c for c in new_cols if c in grid.columns])

n = len(grid)
grid = grid.merge(prac, on="PRACTICE_CODE", how="left", validate="m:1")
assert len(grid) == n

print(prac[["ruc21ind", "imd25_decile"]].isna().sum())

ruc21ind        0
imd25_decile    0
dtype: int64


## Presc

In [8]:
presc = pd.read_parquet(PROCESSED_DIR / "presc_processed.parquet")
presc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3291714 entries, 0 to 3291713
Data columns (total 8 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   YEAR_MONTH                   object        
 1   YEAR_MONTH_DATE              datetime64[ns]
 2   PRACTICE_CODE                object        
 3   BNF_CHEMICAL_SUBSTANCE_CODE  object        
 4   BNF_CHEMICAL_SUBSTANCE       object        
 5   ITEMS                        int64         
 6   NIC                          float64       
 7   ACTUAL_COST                  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 200.9+ MB


In [9]:
# find codes by name (case-insensitive)
print(presc.loc[presc["BNF_CHEMICAL_SUBSTANCE"].str.contains("donepezil", case=False, na=False),
             ["BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"]].drop_duplicates())

# total items per substance, largest first
tot = (presc.groupby(["BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE"])["ITEMS"]
         .sum().sort_values(ascending=False))
print(tot.head(20))

  BNF_CHEMICAL_SUBSTANCE_CODE   BNF_CHEMICAL_SUBSTANCE
9                   0411000D0  Donepezil hydrochloride
BNF_CHEMICAL_SUBSTANCE_CODE  BNF_CHEMICAL_SUBSTANCE             
0411000G0                    Memantine hydrochloride                7677080
0411000D0                    Donepezil hydrochloride                7445348
0409010N0                    Co-careldopa (Carbidopa/levodopa)      5393019
0409010K0                    Co-beneldopa (Benserazide/levodopa)    3599104
0409020S0                    Procyclidine hydrochloride             3398726
0409010H0                    Ropinirole hydrochloride               2449588
0409010W0                    Pramipexole                            2130853
0411000E0                    Rivastigmine                           1484407
0409010Y0                    Rasagiline mesilate                     717839
0409010B0                    Amantadine hydrochloride                497146
0409010Z0                    Rotigotine                          

In [10]:
presc['BNF_CHEMICAL_SUBSTANCE_CODE'].unique()

array(['0409010H0', '0409010K0', '0409010N0', '0409010T0', '0409010V0',
       '0409010W0', '0409010Y0', '0409020C0', '0409020S0', '0411000D0',
       '0411000E0', '0411000F0', '0411000G0', '0409010A0', '0409010Z0',
       '0409010AA', '0409010B0', '0409010X0', '0409010AB', '0409010AC',
       '0409010U0', '0409020N0', '0409010S0', '0409020E0', '0409010D0',
       '0409010I0', '0409020G0'], dtype=object)

In [11]:
import pandas as pd

# df = your DataFrame (3,291,714 rows)
code = "0409010A0"   # <- change to your BNF_CHEMICAL_SUBSTANCE_CODE

sub = presc[presc["BNF_CHEMICAL_SUBSTANCE_CODE"] == code]

if sub.empty:
    print(f"No rows for code {code}")
else:
    print("Substance:", sub["BNF_CHEMICAL_SUBSTANCE"].iloc[0])
    print(f"Total items:       {sub['ITEMS'].sum():,}")
    print(f"Total NIC:         £{sub['NIC'].sum():,.2f}")
    print(f"Total actual cost: £{sub['ACTUAL_COST'].sum():,.2f}")
    print(f"Practices:         {sub['PRACTICE_CODE'].nunique():,}")
    print(f"Date range:        {sub['YEAR_MONTH_DATE'].min():%Y-%m} to {sub['YEAR_MONTH_DATE'].max():%Y-%m}")

    # by month
    monthly = (sub.groupby("YEAR_MONTH_DATE")[["ITEMS", "NIC", "ACTUAL_COST"]]
                  .sum().sort_index())
    print(monthly.to_string())

Substance: Apomorphine hydrochloride
Total items:       28,964
Total NIC:         £17,369,663.43
Total actual cost: £16,674,654.41
Practices:         1,102
Date range:        2022-07 to 2026-03
                 ITEMS        NIC   ACTUAL_COST
YEAR_MONTH_DATE                                
2022-07-01         717  426665.33  400906.12610
2022-08-01         748  434770.10  408198.48069
2022-09-01         717  415618.26  390882.44537
2022-10-01         701  402108.80  377688.43010
2022-11-01         725  421489.24  395836.34463
2022-12-01         731  429973.73  403919.75554
2023-01-01         706  396681.16  373528.44990
2023-02-01         629  382765.73  361324.91013
2023-03-01         735  435722.48  410527.81297
2023-04-01         637  378837.74  359022.72171
2023-05-01         672  397552.68  376458.73730
2023-06-01         703  403543.74  381756.55600
2023-07-01         644  381948.68  359666.06702
2023-08-01         706  415711.68  391170.00064
2023-09-01         672  398359.09  374

In [12]:
presc = pd.read_parquet(PROCESSED_DIR / "presc_processed.parquet")
presc.info()

key = ["PRACTICE_CODE", "YEAR_MONTH_DATE", "BNF_CHEMICAL_SUBSTANCE_CODE"]
vals = ["ITEMS", "NIC", "ACTUAL_COST"]

# --- merge apomorphine: 0409010AC -> 0409010A0 (before summing) ---
presc["BNF_CHEMICAL_SUBSTANCE_CODE"] = (
    presc["BNF_CHEMICAL_SUBSTANCE_CODE"].astype(str).str.strip().replace({"0409010AC": "0409010A0"})
)
assert not (presc["BNF_CHEMICAL_SUBSTANCE_CODE"] == "0409010AC").any()

n_in = len(presc)
presc_slim = presc.groupby(key, as_index=False)[vals].sum()
print(f"Rows: {n_in:,} -> {len(presc_slim):,} after summing")
assert not presc_slim.duplicated(key).any()

# What will not match the grid?
m = presc_slim.merge(grid[key], on=key, how="left", indicator=True)
print(m["_merge"].value_counts())
lost = m[m["_merge"] == "left_only"]
print(f"\nPrescribing rows not in grid: {len(lost):,} of {len(presc_slim):,}")
print("Share of totals lost:")
print((lost[vals].sum() / presc_slim[vals].sum()).map("{:.4%}".format).to_string())
print("Practices affected:", lost["PRACTICE_CODE"].nunique())
print("Substance codes affected:", lost["BNF_CHEMICAL_SUBSTANCE_CODE"].nunique())

n = len(grid)
grid_presc = grid.merge(presc_slim, on=key, how="left", validate="1:1")
assert len(grid_presc) == n

grid_presc[vals] = grid_presc[vals].fillna(0)

# Totals after merge should equal the matched prescribing totals
matched = m[m["_merge"] == "both"][vals].sum()
print("\nGrid total vs matched prescribing total:")
print(pd.DataFrame({"grid": grid_presc[vals].sum(), "matched": matched}).to_string())

# apomorphine check: one code, nothing left under the old one
print("\nApomorphine rows in grid_presc:",
      (grid_presc["BNF_CHEMICAL_SUBSTANCE_CODE"] == "0409010A0").sum())
grid_presc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3291714 entries, 0 to 3291713
Data columns (total 8 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   YEAR_MONTH                   object        
 1   YEAR_MONTH_DATE              datetime64[ns]
 2   PRACTICE_CODE                object        
 3   BNF_CHEMICAL_SUBSTANCE_CODE  object        
 4   BNF_CHEMICAL_SUBSTANCE       object        
 5   ITEMS                        int64         
 6   NIC                          float64       
 7   ACTUAL_COST                  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 200.9+ MB
Rows: 3,291,714 -> 3,291,192 after summing
_merge
both          3251789
left_only       39403
right_only          0
Name: count, dtype: int64

Prescribing rows not in grid: 39,403 of 3,291,192
Share of totals lost:
ITEMS          0.5314%
NIC            0.5574%
ACTUAL_COST    0.5572%
Practices affected: 1639
Substa

## gp_wf and gp_reg

In [13]:
gpwf = pd.read_parquet(PROCESSED_DIR / "gpwf_processed.parquet")
gp_reg = pd.read_parquet(PROCESSED_DIR / "GP_REG_processed.parquet")

In [14]:
key = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]
org_cols = ["ICB_CODE", "ICB_NAME", "SUB_ICB_CODE", "SUB_ICB_NAME", "PCN_CODE", "PCN_NAME"]

# gpwf: one row per practice-month
wf = (
    gpwf.rename(columns={"PRAC_CODE": "PRACTICE_CODE"})
        [key + ["TOTAL_GP_FTE"] + org_cols]
)
assert not wf.duplicated(key).any()

# gp_reg: one row per practice-month, with a column for every sex x age band
reg_long = gp_reg.rename(columns={"ORG_CODE": "PRACTICE_CODE", "EXTRACT_DATE": "YEAR_MONTH_DATE"})

reg = reg_long.pivot_table(
    index=key, columns=["SEX", "AGE_BAND"], values="NUMBER_OF_PATIENTS",
    aggfunc="sum", observed=True, fill_value=0,
)
reg.columns = [
    f"PATIENTS_{sex}_{str(band).replace('-', '_').replace('+', 'PLUS')}"
    for sex, band in reg.columns
]
band_cols = list(reg.columns)

# nothing lost in the pivot
assert reg[band_cols].sum().sum() == reg_long["NUMBER_OF_PATIENTS"].sum()

# drop any existing copies to avoid _x/_y suffixes
new_cols = ["TOTAL_GP_FTE"] + org_cols + [c for c in reg.columns if c not in key]
grid_presc_gp = grid_presc.drop(columns=[c for c in new_cols if c in grid_presc.columns])

n = len(grid_presc_gp)
grid_presc_gp = (
    grid_presc_gp.merge(wf, on=key, how="left", validate="m:1")
        .merge(reg, on=key, how="left", validate="m:1")
)
assert len(grid_presc_gp) == n

cov = ["TOTAL_GP_FTE", band_cols[0]]   # or check all: grid_presc_gp[band_cols].isna().any(axis=1).sum()
print(grid_presc_gp[cov].isna().sum())          # missing before any fill
print(grid_presc_gp[org_cols].isna().sum())
print(f"{len(band_cols)} sex x age-band columns added")

TOTAL_GP_FTE           242580
PATIENTS_FEMALE_0_4    156728
dtype: int64
ICB_CODE        164268
ICB_NAME        164268
SUB_ICB_CODE    163878
SUB_ICB_NAME    164268
PCN_CODE        253032
PCN_NAME        253032
dtype: int64
38 sex x age-band columns added


In [15]:
grid_presc_gp.to_parquet(MERGED_DIR / "grid_presc_gp.parquet", index=False)

# Processing and Cleaning

In [16]:
df_raw = pd.read_parquet(MERGED_DIR / "grid_presc_gp.parquet")
key = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]

pm_raw = (df_raw.groupby(key, as_index=False, observed=True)
          .agg(ITEMS=("ITEMS", "sum"), NIC=("NIC", "sum"), ACTUAL_COST=("ACTUAL_COST", "sum")))
static = (df_raw.drop(columns=["BNF_CHEMICAL_SUBSTANCE_CODE", "BNF_CHEMICAL_SUBSTANCE",
                               "ITEMS", "NIC", "ACTUAL_COST"])
          .drop_duplicates(key))
pm_raw = pm_raw.merge(static, on=key, validate="1:1").sort_values(key).reset_index(drop=True)

## 1. Interpolate

In [17]:
def interp_short_gaps(s, max_gap):
    """Interpolate interior NaN runs of <= max_gap; leave longer runs untouched."""
    filled = s.interpolate(limit_area="inside")
    na = s.isna()
    run_id = (na != na.shift()).cumsum()
    run_len = na.groupby(run_id).transform("size")
    return filled.mask(na & (run_len > max_gap))

MAX_GAP = 3
pm_raw = pm_raw.sort_values(key).reset_index(drop=True)
grp = pm_raw["PRACTICE_CODE"]

# interpolate interior NaNs of <= 3 months in age/sex bands
band_cols = [c for c in pm_raw.columns if c.startswith("PATIENTS_")]

was_na = pm_raw[band_cols].isna().any(axis=1)
pm_raw[band_cols] = (pm_raw.groupby("PRACTICE_CODE", observed=True)[band_cols]
                     .transform(lambda s: interp_short_gaps(s, MAX_GAP)))
still_na = pm_raw[band_cols].isna().any(axis=1)
pm_raw["AGE_SEX_imputed"] = was_na & ~still_na

print(f"Interpolated {pm_raw['AGE_SEX_imputed'].sum():,} practice-months; "
      f"{still_na.sum():,} still missing (leading/trailing or gap > {MAX_GAP})")

# interpolate interior zeros/NaNs of <= 3 months in TOTAL_GP_FTE
fte = pm_raw["TOTAL_GP_FTE"]
valid = fte.where(fte > 0)
prev_v = valid.groupby(grp, observed=True).ffill()
next_v = valid.groupby(grp, observed=True).bfill()
interior_zero = (fte == 0) & prev_v.notna() & next_v.notna()

fte_clean = fte.mask(interior_zero)          # interior zeros -> NaN; edge zeros stay 0
was_na_fte = fte_clean.isna()

pm_raw["TOTAL_GP_FTE"] = (fte_clean.groupby(grp, observed=True)
                          .transform(lambda s: interp_short_gaps(s, MAX_GAP)))
pm_raw["GP_FTE_imputed"] = was_na_fte & pm_raw["TOTAL_GP_FTE"].notna()

print(f"{interior_zero.sum():,} interior zeros set to NaN")
print(f"Interpolated {pm_raw['GP_FTE_imputed'].sum():,} practice-months for TOTAL_GP_FTE; "
      f"{pm_raw['TOTAL_GP_FTE'].isna().sum():,} still NaN")

# push filled bands, FTE and flags back onto the substance-level grid
fill_all = band_cols + ["AGE_SEX_imputed", "TOTAL_GP_FTE", "GP_FTE_imputed"]
df_raw = (df_raw.drop(columns=[c for c in fill_all if c in df_raw.columns])
                .merge(pm_raw[key + fill_all], on=key, how="left", validate="m:1"))

Interpolated 11 practice-months; 6,017 still missing (leading/trailing or gap > 3)
16 interior zeros set to NaN
Interpolated 56 practice-months for TOTAL_GP_FTE; 9,290 still NaN


In [18]:
g = pm_raw.groupby("PRACTICE_CODE", observed=True)
print("PRACTICE-MONTH LEVEL")
print(f"  practice-months: {len(pm_raw):,}")
print(f"  age-sex imputed: {int(pm_raw['AGE_SEX_imputed'].sum()):,} | still missing: {int(pm_raw[band_cols].isna().any(axis=1).sum()):,}")
print(f"  FTE imputed: {int(pm_raw['GP_FTE_imputed'].sum()):,} | still NaN: {int(pm_raw['TOTAL_GP_FTE'].isna().sum()):,} | still zero: {int((pm_raw['TOTAL_GP_FTE'] == 0).sum()):,}")

print("PRACTICE LEVEL")
print(f"  practices: {pm_raw['PRACTICE_CODE'].nunique():,}")
print(f"  with any age-sex imputed: {int((g['AGE_SEX_imputed'].sum() > 0).sum()):,}")
print(f"  with any FTE imputed: {int((g['GP_FTE_imputed'].sum() > 0).sum()):,}")
print(f"  with any FTE still NaN: {int((g['TOTAL_GP_FTE'].apply(lambda s: s.isna().any())).sum()):,}")
print(f"  with any age-sex still missing: {int((g.apply(lambda d: d[band_cols].isna().any(axis=1).any())).sum()):,}")

PRACTICE-MONTH LEVEL
  practice-months: 289,521
  age-sex imputed: 11 | still missing: 6,017
  FTE imputed: 56 | still NaN: 9,290 | still zero: 272
PRACTICE LEVEL
  practices: 6,639
  with any age-sex imputed: 5
  with any FTE imputed: 37
  with any FTE still NaN: 594
  with any age-sex still missing: 454


/var/folders/z0/9vpcvnz51jzd7wwth6rlb98h0000gn/T/ipykernel_67017/3346036394.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(f"  with any age-sex still missing: {int((g.apply(lambda d: d[band_cols].isna().any(axis=1).any())).sum()):,}")


In [19]:
icb_col, icb_name = "ICB_CODE", "ICB_NAME"            # edit icb_name if your column is called something else
pm_raw[icb_col] = pm_raw[icb_col].astype(object)
pm_raw[icb_name] = pm_raw[icb_name].astype(object)

# --- 1. ICB code: fill from the practice's single ICB ---
n_distinct = pm_raw.groupby("PRACTICE_CODE", observed=True)[icb_col].nunique()
print(f"Practices with 0 ICBs (always missing): {(n_distinct == 0).sum():,}")
print(f"Practices with 1 ICB:                   {(n_distinct == 1).sum():,}")
print(f"Practices with >1 ICB (moved):          {(n_distinct > 1).sum():,}")

mode_icb = pm_raw.groupby("PRACTICE_CODE", observed=True)[icb_col].agg(
    lambda s: s.dropna().mode().iloc[0] if s.notna().any() else np.nan
)
single = set(n_distinct[n_distinct == 1].index)

was_na_icb = pm_raw[icb_col].isna()
fill_mask = was_na_icb & pm_raw["PRACTICE_CODE"].isin(single)
pm_raw.loc[fill_mask, icb_col] = pm_raw.loc[fill_mask, "PRACTICE_CODE"].astype(object).map(mode_icb)
pm_raw["ICB_imputed"] = was_na_icb & pm_raw[icb_col].notna()

# --- 2. ICB name: fill from the code (most common name for each code) ---
code_to_name = (pm_raw.dropna(subset=[icb_col, icb_name])
                      .groupby(icb_col)[icb_name]
                      .agg(lambda s: s.mode().iloc[0]))
print(f"\nCodes with more than one name: "
      f"{(pm_raw.dropna(subset=[icb_col, icb_name]).groupby(icb_col)[icb_name].nunique() > 1).sum()}")   # should be 0

was_na_name = pm_raw[icb_name].isna()
pm_raw.loc[was_na_name, icb_name] = pm_raw.loc[was_na_name, icb_col].map(code_to_name)
pm_raw["ICB_NAME_imputed"] = was_na_name & pm_raw[icb_name].notna()

print(f"\nFilled {pm_raw['ICB_imputed'].sum():,} practice-months for ICB_CODE; "
      f"{pm_raw[icb_col].isna().sum():,} still missing")
print(f"Filled {pm_raw['ICB_NAME_imputed'].sum():,} practice-months for ICB_NAME; "
      f"{pm_raw[icb_name].isna().sum():,} still missing")

# --- 3. practices that switch ICB ---
movers = n_distinct[n_distinct > 1].index
display(pm_raw[pm_raw["PRACTICE_CODE"].isin(movers)]
        .groupby("PRACTICE_CODE", observed=True)[icb_col].unique().head(20))

# --- 4. push back to the grid ---
fill_all = [icb_col, "ICB_imputed", icb_name, "ICB_NAME_imputed"]
df_raw = (df_raw.drop(columns=[c for c in fill_all if c in df_raw.columns])
                .merge(pm_raw[key + fill_all], on=key, how="left", validate="m:1"))

Practices with 0 ICBs (always missing): 159
Practices with 1 ICB:                   6,480
Practices with >1 ICB (moved):          0

Codes with more than one name: 0

Filled 3,269 practice-months for ICB_CODE; 3,049 still missing
Filled 3,269 practice-months for ICB_NAME; 3,049 still missing


Series([], Name: ICB_CODE, dtype: object)

## 2. Create columns

In [20]:
# ---- TOTAL_PATIENTS ----
for df in (pm_raw, df_raw):
    df["TOTAL_PATIENTS"] = df[band_cols].sum(axis=1, min_count=1)

print(pm_raw["TOTAL_PATIENTS"].isna().sum())

# ---- Rates ----
dem_codes = ['0411000D0', '0411000E0', '0411000F0', '0411000G0']
dem_rates = {
    ('MALE', '65-69'): 0.012, ('MALE', '70-74'): 0.030, ('MALE', '75-79'): 0.052,
    ('MALE', '80-84'): 0.106, ('MALE', '85-89'): 0.128, ('MALE', '90+'): 0.171,
    ('FEMALE', '65-69'): 0.018, ('FEMALE', '70-74'): 0.025, ('FEMALE', '75-79'): 0.062,
    ('FEMALE', '80-84'): 0.095, ('FEMALE', '85-89'): 0.181, ('FEMALE', '90+'): 0.350,
}
_all = {'20-24': 1.8, '25-29': 1.8, '30-34': 4.5, '35-39': 4.5, '40-44': 14.2, '45-49': 14.2, '90+': 1230.2}
_male = {'50-54': 88.6, '55-59': 159.7, '60-64': 330.1, '65-69': 654.1, '70-74': 1065.7,
         '75-79': 1703.3, '80-84': 2202.2, '85-89': 2221.5}
_female = {'50-54': 48.8, '55-59': 95.8, '60-64': 177.7, '65-69': 318.2, '70-74': 676.3,
           '75-79': 925.7, '80-84': 1314.3, '85-89': 1258.9}
park_rates = {(s, b): r / 100000
              for s, d in [('MALE', {**_all, **_male}), ('FEMALE', {**_all, **_female})]
              for b, r in d.items()}

def band_col(sex, band):
    return f"PATIENTS_{sex}_{band.replace('-', '_').replace('+', 'PLUS')}"

def expected_cases(df, rates):
    cols = {band_col(s, b): r for (s, b), r in rates.items()}
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"Columns not found: {missing}"
    return sum(df[c] * r for c, r in cols.items())

# ---- Drug group ----
codes = df_raw["BNF_CHEMICAL_SUBSTANCE_CODE"].astype(str).str.strip()
is_dem = codes.isin(dem_codes)
assert set(codes[is_dem].unique()) == set(dem_codes), "Dementia codes missing"
df_raw["DRUG_GROUP"] = pd.Categorical(np.where(is_dem, "Dementia", "Parkinson's"))

# ---- Expected cases (per practice-month, assigned by drug group) ----
pm_idx = df_raw.drop_duplicates(key).set_index(key)
df_raw = (df_raw.drop(columns=[c for c in ["_DEM", "_PARK"] if c in df_raw.columns])
                .join(expected_cases(pm_idx, dem_rates).rename("_DEM"), on=key)
                .join(expected_cases(pm_idx, park_rates).rename("_PARK"), on=key))
df_raw["EXPECTED_CASES"] = np.where(is_dem, df_raw["_DEM"], df_raw["_PARK"])
df_raw["EXPECTED_CASES_PER_1000"] = 1000 * df_raw["EXPECTED_CASES"] / df_raw["TOTAL_PATIENTS"]
df_raw["EXPECTED_PREVALENCE"] = df_raw["EXPECTED_CASES"] / df_raw["TOTAL_PATIENTS"]     
df_raw = df_raw.drop(columns=["_DEM", "_PARK"])

print(df_raw[["EXPECTED_CASES", "EXPECTED_CASES_PER_1000", "EXPECTED_PREVALENCE"]].isna().sum())

# ---- Drop age/sex band columns ----
pm_raw = pm_raw.drop(columns=band_cols)
df_raw = df_raw.drop(columns=band_cols)
print(f"Dropped {len(band_cols)} band columns")

6017
EXPECTED_CASES             156442
EXPECTED_CASES_PER_1000    156442
EXPECTED_PREVALENCE        156442
dtype: int64
Dropped 38 band columns


## 3. Drops

In [21]:
cols = ["TOTAL_PATIENTS", "TOTAL_GP_FTE"]
def snap(df): return len(df), df["PRACTICE_CODE"].nunique()
start = snap(pm_raw)

tot_items = pm_raw.groupby("PRACTICE_CODE", observed=True)["ITEMS"].sum()
no_items = tot_items.index[tot_items == 0]

valid = pm_raw[cols].notna() & (pm_raw[cols] > 0)
ever = valid.groupby(pm_raw["PRACTICE_CODE"], observed=True).any()
no_patients = ever.index[~ever["TOTAL_PATIENTS"]]
no_fte = ever.index[~ever["TOTAL_GP_FTE"]]

drop = no_items.union(no_patients).union(no_fte)
df_s1 = df_raw[~df_raw["PRACTICE_CODE"].isin(drop)].copy()
pm_s1 = pm_raw[~pm_raw["PRACTICE_CODE"].isin(drop)].copy()
end = snap(pm_s1)

add_patients = no_patients.difference(no_items)
add_fte = no_fte.difference(no_items).difference(no_patients)

print(f"No items in any month:                          {len(no_items):,}")
print(f"No valid patients in any month (additional):    {len(add_patients):,}")
print(f"No valid GP FTE in any month (additional):      {len(add_fte):,}")
print(f"Dropped {len(drop):,} practices: {start[1]:,} -> {end[1]:,} practices; "
      f"{start[0]-end[0]:,} practice-months removed ({start[0]:,} -> {end[0]:,})")
print(f"(Overlap: no valid patients, any overlap with other tests = {len(no_patients):,}; "
      f"no valid FTE = {len(no_fte):,})")

No items in any month:                          154
No valid patients in any month (additional):    24
No valid GP FTE in any month (additional):      54
Dropped 232 practices: 6,639 -> 6,407 practices; 5,165 practice-months removed (289,521 -> 284,356)
(Overlap: no valid patients, any overlap with other tests = 160; no valid FTE = 201)


In [22]:
cols = ["TOTAL_PATIENTS", "TOTAL_GP_FTE"]

def snap(df):
    return len(df), df["PRACTICE_CODE"].nunique()

start = snap(pm_raw)

# 1. total ITEMS == 0 over the entire period
tot_items = pm_raw.groupby("PRACTICE_CODE", observed=True)["ITEMS"].sum()
no_items = tot_items.index[tot_items == 0]

# 2 and 3. patients / FTE NaN or <= 0 in every month
valid = pm_raw[cols].notna() & (pm_raw[cols] > 0)
ever = valid.groupby(pm_raw["PRACTICE_CODE"], observed=True).any()
no_patients = ever.index[~ever["TOTAL_PATIENTS"]]
no_fte = ever.index[~ever["TOTAL_GP_FTE"]]

drop = no_items.union(no_patients).union(no_fte)

df_s1 = df_raw[~df_raw["PRACTICE_CODE"].isin(drop)].copy()
pm_s1 = pm_raw[~pm_raw["PRACTICE_CODE"].isin(drop)].copy()
end = snap(pm_s1)

only_patients = no_patients.difference(no_items)
only_fte = no_fte.difference(no_items).difference(no_patients)

print(f"No items in any month:                  {len(no_items):,}")
print(f"No valid patients only (not above):     {len(only_patients):,}")
print(f"No valid GP FTE only (not above):       {len(only_fte):,}")
print(f"Dropped {len(drop):,} practices in total: "
      f"{start[1]:,} -> {end[1]:,} practices, {start[0]-end[0]:,} practice-months removed")

print(f"No items in any month:      {len(no_items):,}")
print(f"No valid patients ever:     {len(no_patients):,}")
print(f"No valid GP FTE ever:       {len(no_fte):,}")
print(f"Dropped {len(drop):,} practices in total (a practice can fail more than one test): "
      f"{start[1]:,} -> {end[1]:,} practices, {start[0]-end[0]:,} practice-months removed")

No items in any month:                  154
No valid patients only (not above):     24
No valid GP FTE only (not above):       54
Dropped 232 practices in total: 6,639 -> 6,407 practices, 5,165 practice-months removed
No items in any month:      154
No valid patients ever:     160
No valid GP FTE ever:       201
Dropped 232 practices in total (a practice can fail more than one test): 6,639 -> 6,407 practices, 5,165 practice-months removed


In [23]:
# trim to active window (total_patients > 0 and gp_fte > 0)
active = pm_s1[(pm_s1["TOTAL_PATIENTS"].fillna(0) > 0) & (pm_s1["TOTAL_GP_FTE"].fillna(0) > 0)]
window = active.groupby("PRACTICE_CODE", observed=True)["YEAR_MONTH_DATE"].agg(["min", "max"])

def trim(df):
    first = df["PRACTICE_CODE"].map(window["min"])
    last = df["PRACTICE_CODE"].map(window["max"])
    return df[(df["YEAR_MONTH_DATE"] >= first) & (df["YEAR_MONTH_DATE"] <= last)].copy()

pm_s2, df_s2 = trim(pm_s1), trim(df_s1)

print(f"Trimmed {len(pm_s1) - len(pm_s2):,} practice-months ({len(df_s1) - len(df_s2):,} grid rows)")

Trimmed 4,626 practice-months (120,276 grid rows)


In [24]:
bad = pm_s2.loc[pm_s2[["TOTAL_PATIENTS", "TOTAL_GP_FTE"]].isna().any(axis=1), "PRACTICE_CODE"].unique()

b = len(pm_s2), pm_s2["PRACTICE_CODE"].nunique()
pm_s3 = pm_s2[~pm_s2["PRACTICE_CODE"].isin(bad)].copy()
df_s3 = df_s2[~df_s2["PRACTICE_CODE"].isin(bad)].copy()

print(f"Dropped {len(bad):,} practices with NaN patients or GP FTE "
      f"({b[0] - len(pm_s3):,} practice-months)")

Dropped 42 practices with NaN patients or GP FTE (1,840 practice-months)


In [25]:
nm = pm_s3.groupby("PRACTICE_CODE", observed=True)["YEAR_MONTH_DATE"].nunique()
print(nm.describe())
print(nm.value_counts().sort_index())
for c in [3, 6, 12, 18, 24]:
    print(c, (nm < c).sum())

count    6365.000000
mean       43.659073
std         6.288554
min         1.000000
25%        45.000000
50%        45.000000
75%        45.000000
max        45.000000
Name: YEAR_MONTH_DATE, dtype: float64
YEAR_MONTH_DATE
1       11
2        4
3       13
4        6
5        6
6        4
7        7
8       10
9        9
10      21
11      10
12       8
13      21
14       8
15      12
16      10
17       9
18       1
19       5
20       3
21      15
22      12
23       9
24      13
25      10
26       8
27      11
28       6
29       2
31       3
32       6
33       1
34       7
35       9
36       7
37       3
38       5
39       6
40       4
41       8
42      11
44       5
45    6026
Name: count, dtype: int64
3 15
6 40
12 101
18 169
24 214


In [26]:
MIN_MONTHS = 12
short = nm[nm < MIN_MONTHS].index
b = len(pm_s3), pm_s3["PRACTICE_CODE"].nunique()
pm_s4 = pm_s3[~pm_s3["PRACTICE_CODE"].isin(short)].copy()
df_s4 = df_s3[~df_s3["PRACTICE_CODE"].isin(short)].copy()
print(f"Dropped {len(short):,} practices under {MIN_MONTHS} months, "
      f"{b[0] - len(pm_s4):,} practice-months ({b[1]:,} -> {pm_s4['PRACTICE_CODE'].nunique():,} practices)")

Dropped 101 practices under 12 months, 666 practice-months (6,365 -> 6,264 practices)


In [27]:
def missingness(df, name="df"):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(3),
        "n_unique": df.nunique(dropna=True),
    })
    out = out[out["n_missing"] > 0].sort_values("pct_missing", ascending=False)
    print(f"{name}: {len(df):,} rows, {df.shape[1]} columns, {len(out)} columns with any missing")
    return out

miss = missingness(df_s4, "df_s4") 
display(miss)

df_s4: 7,207,824 rows, 38 columns, 5 columns with any missing


,dtype,n_missing,pct_missing,n_unique
CLOSE_DATE,datetime64[ns],7104786,98.570,14
PCN_CODE,object,52702,0.731,1320
PCN_NAME,object,52702,0.731,1368
SUB_ICB_CODE,object,78,0.001,106
SUB_ICB_NAME,object,78,0.001,106


## 4. Floors

### 4.1 Epidemology

In [28]:
E = "EXPECTED_CASES"
pm = df_raw.drop_duplicates(key + ["DRUG_GROUP"]) if "DRUG_GROUP" in df_raw else df_raw

for g, d in df_raw.groupby("DRUG_GROUP"):
    e = d.drop_duplicates(key)[E]
    print(f"\n{g}")
    print("  missing:", e.isna().sum(), "| <=0:", (e <= 0).sum(), "| <0.5:", (e < 0.5).sum(), "| <1:", (e < 1).sum())
    print(e.describe(percentiles=[.001, .01, .05, .25, .5, .75]).round(3).to_string())

# does prescribing look plausible at low E? (items per expected case, by E decile)
d = df_raw.copy()
d["E_decile"] = pd.qcut(d[E], 10, duplicates="drop")
print(d.groupby(["DRUG_GROUP", "E_decile"], observed=True).apply(
    lambda x: pd.Series({"n_rows": len(x),
                         "share_zero": (x["ITEMS"] == 0).mean(),
                         "items_per_E": x["ITEMS"].sum() / x[E].sum()})).round(3))

/var/folders/z0/9vpcvnz51jzd7wwth6rlb98h0000gn/T/ipykernel_67017/442976706.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, d in df_raw.groupby("DRUG_GROUP"):



Dementia
  missing: 6017 | <=0: 485 | <0.5: 1848 | <1: 2377
count    283504.000
mean        115.271
std          94.575
min           0.000
0.1%          0.000
1%            3.052
5%           19.038
25%          48.851
50%          92.292
75%         154.581
max        1244.795

Parkinson's
  missing: 6017 | <=0: 40 | <0.5: 2150 | <1: 2755
count    283504.000
mean         21.954
std          16.916
min           0.000
0.1%          0.003
1%            1.063
5%            4.409
25%          10.201
50%          18.094
75%          29.090
max         233.617
                                  n_rows  share_zero  items_per_E
DRUG_GROUP  E_decile                                             
Dementia    (-0.001, 6.58]       15188.0       0.916        0.152
            (6.58, 9.743]         4140.0       0.750        0.083
            (9.743, 13.192]       8908.0       0.604        0.139
            (13.192, 16.953]     16760.0       0.558        0.130
            (16.953, 20.946]     23104.0

/var/folders/z0/9vpcvnz51jzd7wwth6rlb98h0000gn/T/ipykernel_67017/442976706.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(d.groupby(["DRUG_GROUP", "E_decile"], observed=True).apply(


In [29]:
grid_clean = df_s4

keys = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]
g = grid_clean 

# one row per practice-month x drug group: offset, total items, and number of substances
pm = (g.groupby(keys + ["DRUG_GROUP"], observed=True)
        .agg(EXPECTED_CASES=("EXPECTED_CASES", "first"),
             ITEMS=("ITEMS", "sum"))
        .reset_index())

edges = [0, 0.5, 1, 2, 3, 4, 5, 7.5, 10, 15, 20, 50, np.inf]
for k, d0 in pm.groupby("DRUG_GROUP", observed=True):
    o = d0["EXPECTED_CASES"]
    big = o >= 20
    ref = d0.loc[big, "ITEMS"].sum() / o[big].sum()     # reference rate where offset is large
    d = d0.assign(bin=pd.cut(o, edges, right=False), zero=d0["ITEMS"] == 0)
    t = (d.groupby("bin", observed=True)
           .agg(rows=("ITEMS", "size"), pct_zero=("zero", "mean"), items=("ITEMS", "sum"),
                off=("EXPECTED_CASES", "sum"), mean=("ITEMS", "mean"), var=("ITEMS", "var")))
    t["pct_zero"] = (100 * t["pct_zero"]).round(1)
    t["rate_vs_ref"] = ((t["items"] / t["off"]) / ref).round(2)   # 1.0 = same as large-offset practices
    t["dispersion"] = (t["var"] / t["mean"]).round(1)
    print(f"\n{k}: reference rate (offset >= 20) = {ref:.4f}")
    print(t[["rows", "pct_zero", "rate_vs_ref", "dispersion"]].to_string())

# cost of candidate floors
print()
for k, ts in [("Dementia", [2, 3, 5, 10, 20]), ("Parkinson's", [1, 2, 3, 5, 10])]:
    d0 = pm[pm["DRUG_GROUP"] == k]
    for t in ts:
        ok = d0["EXPECTED_CASES"] >= t
        print(f"{k} >= {t:>2}: practice-months kept {100*ok.mean():.2f}% | "
              f"practices with any month dropped {d0.loc[~ok, 'PRACTICE_CODE'].nunique()}")


Dementia: reference rate (offset >= 20) = 0.5162
                rows  pct_zero  rate_vs_ref  dispersion
bin                                                    
[0.0, 0.5)       420      86.2         8.40        15.5
[0.5, 1.0)       397      85.9         2.05        15.0
[1.0, 2.0)       283      82.7         0.61         4.0
[2.0, 3.0)        75      65.3         0.53         3.9
[3.0, 4.0)       167      38.3         0.80         2.1
[4.0, 5.0)       221      51.1         0.67         9.9
[5.0, 7.5)       732      30.1         0.79         6.6
[7.5, 10.0)      800      36.2         0.68         6.1
[10.0, 15.0)    3845      11.5         1.03         9.3
[15.0, 20.0)    6292       7.5         1.01         9.1
[20.0, 50.0)   55883       1.7         1.13        15.0
[50.0, inf)   208109       0.1         0.99        48.5

Parkinson's: reference rate (offset >= 20) = 3.0853
                rows  pct_zero  rate_vs_ref  dispersion
bin                                                    
[

In [30]:
FLOORS = {"Dementia": 10, "Parkinson's": 3}
keys = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]

# one row per practice-month, with each group's expected cases as a column
pmg = (grid_clean.groupby(keys + ["DRUG_GROUP"], observed=True)["EXPECTED_CASES"]
       .first().unstack("DRUG_GROUP"))
ok = pd.Series(True, index=pmg.index)
for grp, t in FLOORS.items():
    ok &= pmg[grp] >= t
keep = ok[ok].index.to_frame(index=False)

grid_off = grid_clean.merge(keep, on=keys, how="inner", validate="m:1")

pm_after_floors = grid_off[keys].drop_duplicates().copy()   # <-- add this line

print(f"Practice-months: {len(ok):,} -> {len(keep):,} (-{(~ok).sum():,})")
print(f"Practices: {grid_clean['PRACTICE_CODE'].nunique():,} -> {grid_off['PRACTICE_CODE'].nunique():,}")
print(f"Grid rows: {len(grid_clean):,} -> {len(grid_off):,}")

print(grid_off.groupby("DRUG_GROUP", observed=True)["EXPECTED_CASES"].min().round(2))

after = grid_off.drop_duplicates(keys).groupby("PRACTICE_CODE", observed=True).size()
before = (grid_clean.drop_duplicates(keys).groupby("PRACTICE_CODE", observed=True).size()
          .reindex(after.index))
full = grid_clean["PRACTICE_CODE"].nunique() - len(after)
partial = (after < before).sum()
print(f"Practices fully dropped: {full:,} | partially trimmed: {partial:,}")

Practice-months: 277,224 -> 272,967 (-4,257)
Practices: 6,264 -> 6,189
Grid rows: 7,207,824 -> 7,097,142
DRUG_GROUP
Dementia       10.0
Parkinson's     3.0
Name: EXPECTED_CASES, dtype: float64
Practices fully dropped: 75 | partially trimmed: 84


In [31]:
MIN_MONTHS = 12
nm = grid_off.drop_duplicates(keys).groupby("PRACTICE_CODE", observed=True)["YEAR_MONTH_DATE"].nunique()
short = nm[nm < MIN_MONTHS].index
n_before = grid_off["PRACTICE_CODE"].nunique()
grid_off = grid_off[~grid_off["PRACTICE_CODE"].isin(short)].copy()
print(f"Dropped {len(short):,} practices under {MIN_MONTHS} months "
      f"({n_before:,} -> {grid_off['PRACTICE_CODE'].nunique():,})")

Dropped 14 practices under 12 months (6,189 -> 6,175)


### 4.2 List size

In [32]:
keys = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]
g = grid_off   # offset floors already applied; needs DRUG_GROUP, EXPECTED_CASES, TOTAL_PATIENTS, ITEMS

# one row per practice-month x drug group
pm = (g.groupby(keys + ["DRUG_GROUP"], observed=True)
        .agg(TOTAL_PATIENTS=("TOTAL_PATIENTS", "first"),
             EXPECTED_CASES=("EXPECTED_CASES", "first"),
             ITEMS=("ITEMS", "sum"))
        .reset_index())

edges = [0, 100, 250, 500, 750, 1000, 1500, 2000, 3000, 5000, 10000, np.inf]

for k, d0 in pm.groupby("DRUG_GROUP", observed=True):
    big = d0["TOTAL_PATIENTS"] >= 5000
    ref = d0.loc[big, "ITEMS"].sum() / d0.loc[big, "EXPECTED_CASES"].sum()
    d = d0.assign(bin=pd.cut(d0["TOTAL_PATIENTS"], edges, right=False), zero=d0["ITEMS"] == 0)
    t = d.groupby("bin", observed=True).agg(
        rows=("ITEMS", "size"), pct_zero=("zero", "mean"), items=("ITEMS", "sum"),
        off=("EXPECTED_CASES", "sum"), mean=("ITEMS", "mean"), var=("ITEMS", "var"))
    t["pct_zero"] = (100 * t["pct_zero"]).round(1)
    t["rate_vs_ref"] = ((t["items"] / t["off"]) / ref).round(2)
    t["dispersion"] = (t["var"] / t["mean"]).round(1)
    print(f"\n{k}: reference rate (list >= 5000) = {ref:.4f}")
    print(t[["rows", "pct_zero", "rate_vs_ref", "dispersion"]].to_string())

# list size is the same for both groups, so use one row per practice-month
pml = g.drop_duplicates(keys)
print("\nList size distribution:")
print(pml["TOTAL_PATIENTS"].describe(percentiles=[.001, .01, .05, .25, .5]).round(0).to_string())

print()
for t in [100, 250, 500, 750, 1000, 1500, 2000, 3000]:
    ok = pml["TOTAL_PATIENTS"] >= t
    print(f"list >= {t:>4}: practice-months kept {100*ok.mean():.2f}% | "
          f"practices with any month dropped {pml.loc[~ok, 'PRACTICE_CODE'].nunique()}")




Dementia: reference rate (list >= 5000) = 0.5172
                     rows  pct_zero  rate_vs_ref  dispersion
bin                                                         
[250.0, 500.0)         50       0.0         7.95         7.8
[500.0, 750.0)         35       0.0         5.99         3.1
[1000.0, 1500.0)      257       5.1         2.62       329.7
[1500.0, 2000.0)      920       9.2         0.84        12.5
[2000.0, 3000.0)     9027       3.8         0.92        13.7
[3000.0, 5000.0)    36715       1.8         0.95        18.0
[5000.0, 10000.0)  114189       0.5         1.02        25.3
[10000.0, inf)     111681       0.1         0.99        51.8

Parkinson's: reference rate (list >= 5000) = 3.2272
                     rows  pct_zero  rate_vs_ref  dispersion
bin                                                         
[250.0, 500.0)         50       0.0         1.90         3.7
[500.0, 750.0)         35       0.0         2.63         1.8
[1000.0, 1500.0)      257       1.9        

In [33]:
MIN_LIST = 1500
keys = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]

pmv = grid_off.drop_duplicates(keys)
med = pmv.groupby("PRACTICE_CODE", observed=True)["TOTAL_PATIENTS"].median()
keep_pracs = med[med >= MIN_LIST].index
grid_floor = grid_off[grid_off["PRACTICE_CODE"].isin(keep_pracs)].copy()
print(f"Practices dropped: {len(med) - len(keep_pracs):,}")

Practices dropped: 8


### 4.3 Save

In [34]:
grid_floor.to_parquet(MERGED_DIR / "df_full_floors.parquet", index=False)

## Missingness check

In [35]:
grid_icb = pd.read_parquet(MERGED_DIR / "df_full_floors.parquet")

In [36]:
def missingness(df, name="df"):
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(3),
        "n_unique": df.nunique(dropna=True),
    })
    out = out[out["n_missing"] > 0].sort_values("pct_missing", ascending=False)
    print(f"{name}: {len(df):,} rows, {df.shape[1]} columns, {len(out)} columns with any missing")
    return out

miss = missingness(grid_icb, "grid_icb")  
display(miss)

grid_icb: 7,085,832 rows, 38 columns, 5 columns with any missing


,dtype,n_missing,pct_missing,n_unique
CLOSE_DATE,datetime64[ns],6989190,98.636,14
PCN_CODE,object,46514,0.656,1319
PCN_NAME,object,46514,0.656,1367
SUB_ICB_CODE,object,78,0.001,106
SUB_ICB_NAME,object,78,0.001,106


## 5. Additions for modelling

In [37]:
grid_icb = pd.read_parquet(MERGED_DIR / "df_full_floors.parquet")
grid_icb.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7085832 entries, 0 to 7085831
Data columns (total 38 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   PRACTICE_CODE                object        
 1   PRACTICE_NAME                object        
 2   POSTCODE                     object        
 3   OPEN_DATE                    datetime64[ns]
 4   CLOSE_DATE                   datetime64[ns]
 5   STATUS                       object        
 6   COMMISSIONER                 object        
 7   PRESCRIBING_SETTING          object        
 8   YEAR_MONTH_DATE              datetime64[ns]
 9   BNF_CHEMICAL_SUBSTANCE_CODE  object        
 10  BNF_CHEMICAL_SUBSTANCE       object        
 11  BNF_PARAGRAPH_CODE           object        
 12  BNF_PARAGRAPH                object        
 13  ruc21ind                     string        
 14  sicbl26cd                    string        
 15  msoa21cd                     string        
 16  

In [38]:
# IMD reversal and mean 0sd1
grid_imd = grid_icb.copy()

n_lsoa = 33755
grid_imd["imd25_rev"] = n_lsoa - grid_imd["imd25_rank"] + 1
prac = grid_imd.drop_duplicates("PRACTICE_CODE")["imd25_rev"]

grid_imd["imd25_scaled"] = (grid_imd["imd25_rev"] - prac.mean()) / prac.std()

In [39]:
# creates gpfte per 1000

keys = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]
col = "gp_fte_per_1000"
grid_imd[col] = grid_imd["TOTAL_GP_FTE"] / grid_imd["TOTAL_PATIENTS"] * 1000

pm = grid_imd.drop_duplicates(keys).copy()
pm["pmean"] = pm.groupby("PRACTICE_CODE", observed=True)[col].transform("mean")
pm["within"] = pm[col] - pm["pmean"]

pr = pm.groupby("PRACTICE_CODE", observed=True).agg(
    months=("YEAR_MONTH_DATE", "size"), mean_fte1000=("pmean", "first"),
    mean_patients=("TOTAL_PATIENTS", "mean"), mean_fte=("TOTAL_GP_FTE", "mean"),
    max_abs_within=("within", lambda s: s.abs().max()))

print("Practice means:")
print(pr["mean_fte1000"].describe(percentiles=[.5, .9, .95, .99, .995, .999]).round(3))
for t in [1.5, 2, 2.5, 3]:
    print(f"mean > {t}: {(pr['mean_fte1000'] > t).sum()} practices")

print("\nWithin-practice deviation (raw FTE per 1,000 units):")
print(pm["within"].describe(percentiles=[.001, .01, .99, .999]).round(3))
for t in [0.5, 1, 1.5, 2]:
    print(f"|within| > {t}: {(pm['within'].abs() > t).sum():,} practice-months, "
          f"{pm.loc[pm['within'].abs() > t, 'PRACTICE_CODE'].nunique()} practices")

print("\nTop practice means:")
print(pr.sort_values("mean_fte1000", ascending=False).head(15).round(2))
print("\nTop within-practice swings:")
print(pr.sort_values("max_abs_within", ascending=False).head(15).round(2))

Practice means:
count    6167.000
mean        0.581
std         0.262
min         0.032
50%         0.554
90%         0.907
95%         1.027
99%         1.361
99.5%       1.502
99.9%       1.800
max         4.307
Name: mean_fte1000, dtype: float64
mean > 1.5: 32 practices
mean > 2: 4 practices
mean > 2.5: 2 practices
mean > 3: 2 practices

Within-practice deviation (raw FTE per 1,000 units):
count    272532.000
mean          0.000
std           0.102
min          -1.176
0.1%         -0.513
1%           -0.276
50%          -0.001
99%           0.286
99.9%         0.564
max           1.826
Name: within, dtype: float64
|within| > 0.5: 696 practice-months, 145 practices
|within| > 1: 27 practice-months, 11 practices
|within| > 1.5: 2 practice-months, 2 practices
|within| > 2: 0 practice-months, 0 practices

Top practice means:
               months  mean_fte1000  mean_patients  mean_fte  max_abs_within
PRACTICE_CODE                                                               
L83616    

In [40]:
# drop outliers
THRESH = 2
pm_all = grid_imd.drop_duplicates(keys)
med = pm_all.groupby("PRACTICE_CODE", observed=True)[col].median()
bad = med.index[med > THRESH]
print("Dropping:", list(bad))
grid_imd = grid_imd[~grid_imd["PRACTICE_CODE"].isin(bad)].copy()
print("Practices left:", grid_imd["PRACTICE_CODE"].nunique())

# between/within decomposition on the final sample
pm = grid_imd.drop_duplicates(keys).copy()
pmean = pm.groupby("PRACTICE_CODE", observed=True)[col].mean().rename("gpfte_practice_mean")
pm = pm.join(pmean, on="PRACTICE_CODE")
pm["gpfte_within"] = pm[col] - pm["gpfte_practice_mean"]
mean_b, sd_b, sd_w = pmean.mean(), pmean.std(), pm["gpfte_within"].std()

grid_imd = (grid_imd.drop(columns=["gpfte_practice_mean", "gpfte_within"], errors="ignore")
            .merge(pm[keys + ["gpfte_practice_mean", "gpfte_within"]], on=keys, how="left", validate="m:1"))
grid_imd["gpfte_practice_mean"] = (grid_imd["gpfte_practice_mean"] - mean_b) / sd_b
grid_imd["gpfte_within"] = grid_imd["gpfte_within"] / sd_w
print(f"between mean/SD: {mean_b:.3f}/{sd_b:.3f} | within SD: {sd_w:.3f}")

Dropping: ['E85659', 'L82017', 'L83616', 'M85158']
Practices left: 6163
between mean/SD: 0.580/0.254 | within SD: 0.101


In [41]:
# checks
keys = ["PRACTICE_CODE", "YEAR_MONTH_DATE"]
col  = "gp_fte_per_1000"

# one row per practice-month, after exclusions
pm = grid_imd.drop_duplicates(keys).copy()
pm["pmean_raw"]  = pm.groupby("PRACTICE_CODE", observed=True)[col].transform("mean")
pm["within_raw"] = pm[col] - pm["pmean_raw"]

# SDs in raw FTE per 1,000 patients
pr_means = pm.groupby("PRACTICE_CODE", observed=True)["pmean_raw"].first()
sd_b_raw = pr_means.std()
sd_w_raw = pm["within_raw"].std()
print(f"Between SD (raw FTE/1000): {sd_b_raw:.4f}")
print(f"Within  SD (raw FTE/1000): {sd_w_raw:.4f}")
print(f"Ratio between/within:      {sd_b_raw / sd_w_raw:.2f}")

# check the scaled variables used in the models
chk = pm[["PRACTICE_CODE", "YEAR_MONTH_DATE"]].merge(
    grid_imd.drop_duplicates(keys)[keys + ["gpfte_practice_mean", "gpfte_within"]], on=keys)
print("\nScaled variables (should be ~mean 0, SD 1):")
print("gpfte_practice_mean  mean/SD (row-level):",
      round(chk["gpfte_practice_mean"].mean(), 4), "/", round(chk["gpfte_practice_mean"].std(), 4))
print("gpfte_practice_mean  mean/SD (per practice):",
      round(chk.groupby("PRACTICE_CODE", observed=True)["gpfte_practice_mean"].first().mean(), 4), "/",
      round(chk.groupby("PRACTICE_CODE", observed=True)["gpfte_practice_mean"].first().std(), 4))
print("gpfte_within         mean/SD:",
      round(chk["gpfte_within"].mean(), 4), "/", round(chk["gpfte_within"].std(), 4))

# within-practice mean should be ~0 for every practice
wm = chk.groupby("PRACTICE_CODE", observed=True)["gpfte_within"].mean().abs()
print("max |practice mean of gpfte_within|:", round(wm.max(), 6))

# extremes in within-SD units
print("\nwithin (SD units) min/max:", round(chk["gpfte_within"].min(), 2), "/", round(chk["gpfte_within"].max(), 2))
print("share |within| > 5 SD:", round((chk["gpfte_within"].abs() > 5).mean() * 100, 3), "%")

Between SD (raw FTE/1000): 0.2537
Within  SD (raw FTE/1000): 0.1012
Ratio between/within:      2.51

Scaled variables (should be ~mean 0, SD 1):
gpfte_practice_mean  mean/SD (row-level): 0.0093 / 0.9973
gpfte_practice_mean  mean/SD (per practice): -0.0 / 1.0
gpfte_within         mean/SD: 0.0 / 1.0
max |practice mean of gpfte_within|: 0.0

within (SD units) min/max: -11.62 / 18.04
share |within| > 5 SD: 0.23 %


In [42]:
p = grid_imd.drop_duplicates(keys)
print(p.groupby("PRACTICE_CODE", observed=True)["gpfte_practice_mean"].first().agg(["mean", "std"]).round(3))
print(p["gpfte_within"].agg(["mean", "std"]).round(3))

mean   -0.0
std     1.0
Name: gpfte_practice_mean, dtype: float64
mean    0.0
std     1.0
Name: gpfte_within, dtype: float64


In [43]:
# rurlity collapse to binary
urban_categories = ['UF1', 'UN1']

grid_imd["rural"] = (~grid_imd["ruc21ind"].astype(str).isin(urban_categories)).astype(int)

In [44]:
# time controls

grid_imd["calendar_quarter"] = ((grid_imd["YEAR_MONTH_DATE"].dt.month - 1) // 3 + 1).astype(str)

grid_imd["month_index"] = grid_imd["YEAR_MONTH_DATE"].dt.year * 12 + grid_imd["YEAR_MONTH_DATE"].dt.month
grid_imd["month_index"] -= grid_imd["month_index"].min()

In [45]:
grid_imd.columns

Index(['PRACTICE_CODE', 'PRACTICE_NAME', 'POSTCODE', 'OPEN_DATE', 'CLOSE_DATE',
       'STATUS', 'COMMISSIONER', 'PRESCRIBING_SETTING', 'YEAR_MONTH_DATE',
       'BNF_CHEMICAL_SUBSTANCE_CODE', 'BNF_CHEMICAL_SUBSTANCE',
       'BNF_PARAGRAPH_CODE', 'BNF_PARAGRAPH', 'ruc21ind', 'sicbl26cd',
       'msoa21cd', 'lsoa21cd', 'imd25_rank', 'imd25_decile', 'ITEMS', 'NIC',
       'ACTUAL_COST', 'SUB_ICB_CODE', 'SUB_ICB_NAME', 'PCN_CODE', 'PCN_NAME',
       'AGE_SEX_imputed', 'TOTAL_GP_FTE', 'GP_FTE_imputed', 'ICB_CODE',
       'ICB_imputed', 'ICB_NAME', 'ICB_NAME_imputed', 'TOTAL_PATIENTS',
       'DRUG_GROUP', 'EXPECTED_CASES', 'EXPECTED_CASES_PER_1000',
       'EXPECTED_PREVALENCE', 'imd25_rev', 'imd25_scaled', 'gp_fte_per_1000',
       'gpfte_practice_mean', 'gpfte_within', 'rural', 'calendar_quarter',
       'month_index'],
      dtype='object')

In [46]:
grid_imd.to_parquet((FINAL_DIR / "df_full_modelling.parquet"), index=False)

# Build data for modelling

In [47]:
df_full = pd.read_parquet(FINAL_DIR / "df_full_modelling.parquet")
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7081724 entries, 0 to 7081723
Data columns (total 46 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   PRACTICE_CODE                object        
 1   PRACTICE_NAME                object        
 2   POSTCODE                     object        
 3   OPEN_DATE                    datetime64[ns]
 4   CLOSE_DATE                   datetime64[ns]
 5   STATUS                       object        
 6   COMMISSIONER                 object        
 7   PRESCRIBING_SETTING          object        
 8   YEAR_MONTH_DATE              datetime64[ns]
 9   BNF_CHEMICAL_SUBSTANCE_CODE  object        
 10  BNF_CHEMICAL_SUBSTANCE       object        
 11  BNF_PARAGRAPH_CODE           object        
 12  BNF_PARAGRAPH                object        
 13  ruc21ind                     string        
 14  sicbl26cd                    string        
 15  msoa21cd                     string        
 16  

In [48]:
print(f"The final dataset contains {df_full['PRACTICE_CODE'].nunique():,} practices, "
      f"{df_full['BNF_CHEMICAL_SUBSTANCE_CODE'].nunique():,} substances and "
      f"{df_full['YEAR_MONTH_DATE'].nunique():,} months giving {len(df_full):,} rows.")

The final dataset contains 6,163 practices, 26 substances and 45 months giving 7,081,724 rows.


In [49]:
KEEP = ["PRACTICE_CODE", "ICB_CODE", "YEAR_MONTH_DATE", "ITEMS", "EXPECTED_CASES",
        "imd25_scaled", "imd25_decile","rural", "gpfte_practice_mean", "gpfte_within",
        "calendar_quarter", "month_index"]
KEYS = ["PRACTICE_CODE", "ICB_CODE", "YEAR_MONTH_DATE"]

codes_all = df_full["BNF_CHEMICAL_SUBSTANCE_CODE"].astype(str).str.strip()


def build_2level_data_pooled(codes, group_name):
    df_sub = df_full.loc[codes_all.isin(codes), KEEP].copy()
    assert len(df_sub) > 0, f"No rows for {group_name}"

    df_sub = df_sub.dropna(subset=["ICB_CODE"])
    df_sub["PRACTICE_CODE"] = df_sub["PRACTICE_CODE"].astype(str)
    df_sub["ICB_CODE"] = df_sub["ICB_CODE"].astype(str)

    agg = {c: "first" for c in KEEP if c not in KEYS}
    agg["ITEMS"] = "sum"
    df_sub = df_sub.groupby(KEYS, as_index=False, observed=True).agg(agg)

    df_sub = df_sub.dropna()
    df_sub = df_sub[df_sub["EXPECTED_CASES"] > 0]
    df_sub["y_positive"] = (df_sub["ITEMS"] > 0).astype(int)
    print(f"[{group_name}] Rows: {len(df_sub):,}")
    return df_sub


def build_2level_data_pooled_cached(codes, group_name, subdir, rebuild=False):
    folder = FINAL_DIR / subdir
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{group_name}_2level.parquet"
    if path.exists() and not rebuild:
        return pd.read_parquet(path)
    df = build_2level_data_pooled(codes, group_name)
    df.to_parquet(path, index=False)
    return df

In [50]:
classes = {
    "Dementia": {
        "AChEI": {
            "Donepezil":    ["0411000D0"],
            "Rivastigmine": ["0411000E0"],
            "Galantamine":  ["0411000F0"],
        },
        "DEM_NDMA_Receptor_Agonists": {
            "Memantine": ["0411000G0"],
        },
    },
    "Parkinson's": {
        "Anticholinergic": {
            "Benzatropine":    ["0409020E0"],
            "Trihexyphenidyl": ["0409020C0"],
            "Procyclidine":    ["0409020S0"],
            "Orphenadrine":    ["0409020N0"],
            "Biperiden":       ["0409020G0"],
        },
        "Dopamine_Precursor": {
            "Co_beneldopa":                  ["0409010K0"],
            "Co_careldopa":                  ["0409010N0"],
            "Levodopa_carbidopa_entacapone": ["0409010X0"],
            "Levodopa":                      ["0409010I0"],
            "Carbidopa":                     ["0409010D0"],
        },
        "Dopamine_Agonists": {
            "Cabergoline": ["0409010U0"], #ergot
            "Apomorphine": ["0409010A0"],
            "Pramipexole": ["0409010W0"],
            "Ropinirole":  ["0409010H0"],
            "Rotigotine":  ["0409010Z0"],
        },
        "COMT_Inhibitors": {
            "Entacapone": ["0409010V0"],
            "Opicapone":  ["0409010AB"],
            "Tolcapone":  ["0409010S0"],
        },
        "MAOB_Inhibitors": {
            "Rasagiline": ["0409010Y0"],
            "Safinamide": ["0409010AA"],
            "Selegiline": ["0409010T0"],
        },
        "PD_NDMA_Receptor_Agonists": {
            "Amantadine": ["0409010B0"],
        },
    }
}

# every substance, used for the data check only
all_substances = {s: c for g in classes.values() for cl in g.values() for s, c in cl.items()}

# substances that get their own individual model: skip classes with a single substance,
# since the class-level model already covers them
substances = {s: c for g in classes.values() for cl in g.values() if len(cl) > 1 for s, c in cl.items()}

codes_by_group = {g: [c for cl in d.values() for cs in cl.values() for c in cs] for g, d in classes.items()}
codes_by_class = {(g, k): [c for cs in cl.values() for c in cs] for g, d in classes.items() for k, cl in d.items()}

# check against the data so nothing is silently dropped
in_grid = set(df_full["BNF_CHEMICAL_SUBSTANCE_CODE"].astype(str).str.strip())
listed = {c for cs in all_substances.values() for c in cs}            # use all_substances here
print("In grid, not listed:", sorted(in_grid - listed))
print("Listed, not in grid:", sorted(listed - in_grid))
print("Individual models:", len(substances), "| skipped:", sorted(set(all_substances) - set(substances)))

In grid, not listed: []
Listed, not in grid: []
Individual models: 24 | skipped: ['Amantadine', 'Memantine']


In [51]:
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# level -> {name: codes}
levels = {
    "disease":    {g.replace("'", "").replace(" ", "_"): codes for g, codes in codes_by_group.items()},
    "drug_type":  {k: codes for (g, k), codes in codes_by_class.items()},
    "individual": dict(substances),
}

# Safety: one offset per model, so every set must sit inside a single drug group
group_of = {c: g for g, cs in codes_by_group.items() for c in cs}
for level, sets in levels.items():
    for name, codes in sets.items():
        assert len({group_of[c] for c in codes}) == 1, f"{level}/{name} mixes drug groups"

for level, sets in levels.items():
    for name, codes in sets.items():
        build_2level_data_pooled_cached(codes, name, subdir=level, rebuild=True)

[Dementia] Rows: 272,374
[Parkinsons] Rows: 272,374
[AChEI] Rows: 272,374
[DEM_NDMA_Receptor_Agonists] Rows: 272,374
[Anticholinergic] Rows: 272,374
[Dopamine_Precursor] Rows: 272,374
[Dopamine_Agonists] Rows: 272,374
[COMT_Inhibitors] Rows: 272,374
[MAOB_Inhibitors] Rows: 272,374
[PD_NDMA_Receptor_Agonists] Rows: 272,374
[Donepezil] Rows: 272,374
[Rivastigmine] Rows: 272,374
[Galantamine] Rows: 272,374
[Benzatropine] Rows: 272,374
[Trihexyphenidyl] Rows: 272,374
[Procyclidine] Rows: 272,374
[Orphenadrine] Rows: 272,374
[Biperiden] Rows: 272,374
[Co_beneldopa] Rows: 272,374
[Co_careldopa] Rows: 272,374
[Levodopa_carbidopa_entacapone] Rows: 272,374
[Levodopa] Rows: 272,374
[Carbidopa] Rows: 272,374
[Cabergoline] Rows: 272,374
[Apomorphine] Rows: 272,374
[Pramipexole] Rows: 272,374
[Ropinirole] Rows: 272,374
[Rotigotine] Rows: 272,374
[Entacapone] Rows: 272,374
[Opicapone] Rows: 272,374
[Tolcapone] Rows: 272,374
[Rasagiline] Rows: 272,374
[Safinamide] Rows: 272,374
[Selegiline] Rows: 272

# .tex table

In [52]:
df_full = pd.read_parquet(FINAL_DIR / "df_full_modelling.parquet")
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7081724 entries, 0 to 7081723
Data columns (total 46 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   PRACTICE_CODE                object        
 1   PRACTICE_NAME                object        
 2   POSTCODE                     object        
 3   OPEN_DATE                    datetime64[ns]
 4   CLOSE_DATE                   datetime64[ns]
 5   STATUS                       object        
 6   COMMISSIONER                 object        
 7   PRESCRIBING_SETTING          object        
 8   YEAR_MONTH_DATE              datetime64[ns]
 9   BNF_CHEMICAL_SUBSTANCE_CODE  object        
 10  BNF_CHEMICAL_SUBSTANCE       object        
 11  BNF_PARAGRAPH_CODE           object        
 12  BNF_PARAGRAPH                object        
 13  ruc21ind                     string        
 14  sicbl26cd                    string        
 15  msoa21cd                     string        
 16  

In [53]:
RAW_FILE   = BASE_DIR / "Data" / "Merged" / "grid_presc_gp.parquet"
FINAL_FILE = BASE_DIR / "Data" / "Final" / "df_full_modelling.parquet"
OUT_DIR    = BASE_DIR / "Tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

KEY_P, KEY_M = "PRACTICE_CODE", "YEAR_MONTH_DATE"
N_LSOA = 33755
URBAN  = ["UF1", "UN1"]

# ---- 1. load raw practice-months (only the columns needed) ----
names = pq.read_schema(RAW_FILE).names
bands = [c for c in names if c.startswith("PATIENTS_")]
raw = pq.read_table(RAW_FILE, columns=[KEY_P, KEY_M, "imd25_rank", "ruc21ind", "TOTAL_GP_FTE"] + bands).to_pandas()
raw = raw.drop_duplicates([KEY_P, KEY_M]).copy()
raw[KEY_P] = raw[KEY_P].astype(str)

pat = raw[bands].sum(axis=1, min_count=1).astype("float64")
fte = pd.to_numeric(raw["TOTAL_GP_FTE"], errors="coerce").astype("float64")
raw["fte1000"] = np.where((pat > 0) & (fte > 0), fte / pat * 1000, np.nan)
raw["pat_pos"] = pat.where(pat > 0)

# ---- 2. excluded = in raw data but not in the final modelling file ----
final_p = set(pd.read_parquet(FINAL_FILE, columns=[KEY_P])[KEY_P].astype(str))
g = raw.groupby(KEY_P)
static = raw.drop_duplicates(KEY_P).set_index(KEY_P)
ruc = static["ruc21ind"].astype("object")

prac = pd.DataFrame({
    "imd_rev":   N_LSOA - static["imd25_rank"].to_numpy(dtype="float64", na_value=np.nan) + 1,
    "rural":     np.where(ruc.isna(), np.nan, (~ruc.astype(str).isin(URBAN)).astype(float)),
    "gpfte":     g["fte1000"].mean(),
    "list_size": g["pat_pos"].median()}, index=static.index).reset_index()
prac["excluded"] = (~prac[KEY_P].isin(final_p)).astype(int)
for c in ["imd_rev", "rural", "gpfte", "list_size"]:
    prac[c] = prac[c].astype("float64")

mu, sd = prac.loc[prac.excluded == 0, "imd_rev"].agg(["mean", "std"])
prac["imd"] = (prac["imd_rev"] - mu) / sd
prac["log_list"] = np.log(prac["list_size"])

n_ex, n_kp, n_all = int(prac.excluded.sum()), int((prac.excluded == 0).sum()), len(prac)
print(f"Practices: {n_all:,} | excluded {n_ex:,} | retained {n_kp:,}")

# ---- 3. descriptive comparison ----
LAB = {"imd": "IMD25 (standardised)", "rural": "Rural practice", "gpfte": "GP FTE per 1,000 patients",
       "list_size": "List size (patients)"}
fp = lambda p: "$<$0.001" if p < 0.001 else f"{p:.3f}"
ex = prac["excluded"].eq(1)
rows, desc = [], {}
for v in LAB:
    x = prac[v]; a, b = x[ex].dropna().to_numpy(), x[~ex].dropna().to_numpy()
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    sdiff = (a.mean() - b.mean()) / pooled
    if v == "rural":
        p = stats.chi2_contingency(pd.crosstab(prac["excluded"], x))[1]
        s_a, s_b = f"{100*a.mean():.1f}\\%", f"{100*b.mean():.1f}\\%"
    else:
        p = stats.mannwhitneyu(a, b).pvalue
        f = (lambda z: f"{np.median(z):.2f} [{np.percentile(z,25):.2f}, {np.percentile(z,75):.2f}]") if v in ("imd", "gpfte") \
            else (lambda z: f"{np.median(z):,.0f} [{np.percentile(z,25):,.0f}, {np.percentile(z,75):,.0f}]")
        s_a, s_b = f(a), f(b)
    desc[v] = dict(a=s_a, b=s_b, sdiff=sdiff, p=p, miss=100 * x[ex].isna().mean())
    rows.append(f"{LAB[v]} & {s_a} & {s_b} & {sdiff:.2f} & {fp(p)} & {100*x[ex].isna().mean():.0f}\\% \\\\")

# ---- 4. adjusted odds of exclusion (per 1 SD; rural = rural vs urban) ----
def logit(cols):
    d = prac.dropna(subset=cols).copy()
    for c in cols:
        if c != "rural": d[c] = (d[c] - d[c].mean()) / d[c].std()
    f = smf.logit("excluded ~ " + " + ".join(cols), data=d).fit(disp=0)
    ci = f.conf_int()
    return d, pd.DataFrame({"OR": np.exp(f.params), "lo": np.exp(ci[0]), "hi": np.exp(ci[1]), "p": f.pvalues}).drop("Intercept")

d1, m1 = logit(["imd", "rural"])
d2, m2 = logit(["imd", "rural", "gpfte", "log_list"])
LAB2 = {"imd": "IMD25 (per 1 SD)", "rural": "Rural vs urban", "gpfte": "GP FTE per 1,000 (per 1 SD)",
        "log_list": "Log list size (per 1 SD)"}
orc = lambda m, v: (f"{m.loc[v,'OR']:.2f} [{m.loc[v,'lo']:.2f}, {m.loc[v,'hi']:.2f}]{'' if m.loc[v,'p']>=0.05 else '*'}"
                    if v in m.index else "--")
rows2 = [f"{LAB2[v]} & {orc(m1, v)} & {orc(m2, v)} \\\\" for v in LAB2]

# ---- 5. write tex ----
table = "\n".join([
 r"\begin{table}[htbp]", r"\centering",
 r"\caption{Characteristics of excluded and retained GP practices}", r"\label{tab:exclusion_descriptives}",
 r"\footnotesize", r"\renewcommand{\arraystretch}{1.2}",
 r"\begin{tabular}{lccccc}", r"\toprule",
 rf" & Excluded ($n$ = {n_ex:,}) & Retained ($n$ = {n_kp:,}) & Std.\ diff. & $p$ & Missing (excluded) \\", r"\midrule",
 *rows, r"\bottomrule", r"\end{tabular}", r"\par\vspace{4pt}",
 r"{\footnotesize \textit{Notes:} Median [IQR] for continuous variables. Std.\ diff.\ is the standardised mean difference "
 r"(excluded minus retained). $p$ from Mann--Whitney (continuous) or $\chi^2$ (rural) tests. "
 r"Covariates are taken from the raw practice data before cleaning.\par}", r"\end{table}", "",
 r"\begin{table}[htbp]", r"\centering",
 r"\caption{Adjusted odds of practice exclusion}", r"\label{tab:exclusion_logit}",
 r"\footnotesize", r"\renewcommand{\arraystretch}{1.2}",
 r"\begin{tabular}{lcc}", r"\toprule",
 rf" & Model 1 ($n$ = {len(d1):,}) & Model 2 ($n$ = {len(d2):,}) \\", r"\midrule",
 *rows2, r"\bottomrule", r"\end{tabular}", r"\par\vspace{4pt}",
 r"{\footnotesize \textit{Notes:} Odds ratios [95\% CI] from logistic regression of exclusion. "
 r"Model 2 uses complete cases only. * $p\geq0.05$.\par}", r"\end{table}"])
(OUT_DIR / "exclusion_check_table.tex").write_text(table + "\n")

text = (rf"Of {n_all:,} practices in the raw data, {n_ex:,} ({100*n_ex/n_all:.1f}\%) were excluded during cleaning. "
        rf"Excluded practices were smaller (median list size {desc['list_size']['a'].split(' [')[0]} vs {desc['list_size']['b'].split(' [')[0]}; "
        rf"standardised difference {desc['list_size']['sdiff']:.2f}), more deprived (standardised difference {desc['imd']['sdiff']:.2f}) "
        rf"and less often rural ({desc['rural']['a']} vs {desc['rural']['b']}). "
        rf"After adjustment for rurality, GP FTE and list size, the odds of exclusion per 1 SD of IMD25 were "
        rf"{m2.loc['imd','OR']:.2f} (95\% CI {m2.loc['imd','lo']:.2f} to {m2.loc['imd','hi']:.2f}), "
        rf"while list size remained strongly associated (OR {m2.loc['log_list','OR']:.2f} per 1 SD of log list size). "
        rf"Selection therefore appears to reflect practice size more than deprivation.")
(OUT_DIR / "exclusion_check_text.tex").write_text(text + "\n")

print(text)
print("\nWritten to", OUT_DIR)
print(r"\input{tables/exclusion_check_table.tex}")
print(r"\input{tables/exclusion_check_text.tex}")

Practices: 6,639 | excluded 476 | retained 6,163
Of 6,639 practices in the raw data, 476 (7.2\%) were excluded during cleaning. Excluded practices were smaller (median list size 3,086 vs 8,751; standardised difference -0.90), more deprived (standardised difference 0.21) and less often rural (8.4\% vs 15.3\%). After adjustment for rurality, GP FTE and list size, the odds of exclusion per 1 SD of IMD25 were 1.13 (95\% CI 0.98 to 1.32), while list size remained strongly associated (OR 0.28 per 1 SD of log list size). Selection therefore appears to reflect practice size more than deprivation.

Written to /Users/trinityrose/Desktop/TEST/Tables
\input{tables/exclusion_check_table.tex}
\input{tables/exclusion_check_text.tex}
